# Module 6 project-exercise demo — Trainable normalizers and Q/K reduction axes

**Implementation demo, not a completed full-scale project.** Synthetic regression with matched data, initial linear layers and optimizer, plus a held-out set. RMSNorm has a gain; DyT/DyISRU also have bias and a shape scalar. Their parameter counts differ and are printed. This is a teaching experiment, not a language-model comparison.
All inputs are small, deterministic, offline, and CPU-friendly. The goal is to
validate interfaces, metrics, invariants, and report structure before spending
GPU or API budget.

In [ ]:
from __future__ import annotations
import importlib.util, subprocess, sys
for package in ("numpy", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])


import math
import random
import time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np

SEED = 20260801
random.seed(SEED)
np.random.seed(SEED)
print(f"reproducible smoke tier; seed={SEED}")
DEMO_LANGUAGE = 'en'

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch"])


## Demo 1. Comparing trainable normalizers

The target is sin(x₀)+0.2x₁ given 32 input features. Use 512 examples for training and reserve 256 for validation. All variants start with identical linear layers. Changing the normalizer also changes its parameter count, which the code reports alongside the error.

Track both curves: fitting the training set and generalizing to unseen examples are different questions. One learning rate and one seed provide an experiment to inspect, rather than a ranking of architectures.

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
torch.set_num_threads(1)
ru = DEMO_LANGUAGE == "ru"
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=["#5E81AC", "#A3BE8C", "#B48EAD"])})
data_rng = torch.Generator().manual_seed(SEED)
x = torch.randn(768, 32, generator=data_rng) * torch.linspace(.5, 3., 32)
y = torch.sin(x[:, 0]) + .2*x[:, 1]
x_train, x_valid = x[:512], x[512:]
y_train, y_valid = y[:512], y[512:]

class RMSNorm(nn.Module):
    def __init__(self):
        super().__init__()
        self.gain = nn.Parameter(torch.ones(32))
    def forward(self, z):
        return self.gain * z * torch.rsqrt(z.square().mean(-1, keepdim=True)+1e-6)

class DyT(nn.Module):
    def __init__(self):
        super().__init__()
        self.alpha = nn.Parameter(torch.tensor(.5))
        self.gain = nn.Parameter(torch.ones(32))
        self.bias = nn.Parameter(torch.zeros(32))
    def forward(self, z):
        return self.gain * torch.tanh(self.alpha*z) + self.bias

class DyISRU(nn.Module):
    def __init__(self):
        super().__init__()
        # b=4 makes the initial slope 1/sqrt(b)=0.5, matching DyT.
        self.raw_b = nn.Parameter(torch.log(torch.expm1(torch.tensor(4.))))
        self.gain = nn.Parameter(torch.ones(32))
        self.bias = nn.Parameter(torch.zeros(32))
    def forward(self, z):
        b = F.softplus(self.raw_b) + 1e-8
        return self.gain * z * torch.rsqrt(b+z.square()) + self.bias

def train(norm_cls):
    torch.manual_seed(SEED)  # These normalizers consume no random draws.
    model = nn.Sequential(nn.Linear(32,32), norm_cls(), nn.SiLU(), nn.Linear(32,1))
    initial = [p.detach().clone() for i in (0,3) for p in model[i].parameters()]
    opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
    history = []
    for _ in range(80):
        model.train()
        loss = F.mse_loss(model(x_train).squeeze(-1), y_train)
        assert torch.isfinite(loss)
        opt.zero_grad(); loss.backward()
        assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())
        opt.step()
        model.eval()
        with torch.no_grad():
            train_mse = F.mse_loss(model(x_train).squeeze(-1), y_train)
            valid = F.mse_loss(model(x_valid).squeeze(-1), y_valid)
        history.append((train_mse.item(),valid.item()))
    return np.asarray(history),initial,sum(p.numel() for p in model.parameters())

fig,axes=plt.subplots(1,2,figsize=(11,4.8),dpi=150)
initial_reference = None
for cls in (RMSNorm,DyT,DyISRU):
    history,initial,count=train(cls)
    if initial_reference is None: initial_reference=initial
    for a,b in zip(initial,initial_reference): torch.testing.assert_close(a,b,atol=0,rtol=0)
    assert np.isfinite(history).all()
    for index,ax in enumerate(axes): ax.plot(np.arange(1,81),history[:,index],label=cls.__name__)
    print(cls.__name__, "parameters:",count,"final train/validation MSE:",history[-1])
for ax,title in zip(axes,["Обучение","Отложенная выборка"] if ru else ["Training","Held-out validation"]):
    ax.set_title(title,loc="left");ax.set_yscale("log")
    ax.set_xlabel("Шаг" if ru else "Step");ax.set_ylabel("MSE");ax.grid(alpha=.25);ax.legend()
fig.tight_layout();plt.show();plt.close(fig)

Read the training–validation gap together with the parameter counts. The initial linear weights are checked for exact equality, but a single learning rate may favor one variant. Repeat with several rates and seeds before making a broader claim, and reserve an independent test set if validation guides that selection.

## Demo 2. Does changing one head affect another?

Evaluate the same query–key pairs with raw vectors, RMS-normalized vectors, and unit L2 vectors. These paired scores are not a complete attention matrix. At head width 64, the RMS scores should be close to eight times the cosine; positive epsilon accounts for a small discrepancy. Next enlarge only head 1 and predict whether head 0 changes under independent head norms versus one shared projection norm.

In [ ]:
# Paired scores isolate scale; they are not a complete attention matrix.
torch.manual_seed(SEED)
q = torch.randn(256, 8, 64) * 2.5
k = torch.randn(256, 8, 64) * 2.5
raw = (q*k).sum(-1)/math.sqrt(64)
q_n=F.rms_norm(q,[64],eps=1e-6);k_n=F.rms_norm(k,[64],eps=1e-6)
rms_scores=(q_n*k_n).sum(-1)/math.sqrt(64)
cosine=(F.normalize(q,dim=-1)*F.normalize(k,dim=-1)).sum(-1)
# A learned logit scale is intentionally omitted from this diagnostic.
torch.testing.assert_close(rms_scores,math.sqrt(64)*cosine,atol=2e-6,rtol=2e-6)
assert rms_scores.std() < raw.std()
print("raw / RMS / cosine std:",*[float(z.std()) for z in (raw,rms_scores,cosine)])
changed=q.clone();changed[:,1]*=10
per_head=F.rms_norm(changed,[64],eps=1e-6)
joint=F.rms_norm(q.flatten(-2),[8*64],eps=1e-6).reshape_as(q)
joint_changed=F.rms_norm(changed.flatten(-2),[8*64],eps=1e-6).reshape_as(q)
torch.testing.assert_close(per_head[:,0],q_n[:,0],atol=0,rtol=0)
assert not torch.allclose(joint[:,0],joint_changed[:,0])
print("Ось нормы меняет связь между головами." if ru else "The reduction axis changes coupling between heads.")

The unchanged head retains its output under a separate head norm. A whole-projection norm changes its denominator because another head grew. This isolates the reduction axis; narrower synthetic score distributions alone do not establish better attention or translation quality.

## Scaling to the full project

For the language-model project, fix tokenization and data order, repeat across seeds and sweep learning rates. Compare quality, gradients, time and memory under a matched budget. For real Q/K, record revision, layer, reduction axis, learned gains and placement relative to RoPE; this demo uses only synthetic vectors.

### Minimum report

1. Data, model, and code revisions, plus the random seed.
2. The primary metric, plus quality, cost, latency, and memory where applicable.
3. A baseline, a matched budget, and at least one negative control.
4. Failed runs and the limits of the supported conclusions.